In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# -------------------------------
# 1. Recalculate Population Metrics
# -------------------------------

# Total population from the three age groups
df['total_population_2024'] = df['2024_pop_youth'] + df['2024_pop_workingage'] + df['2024_pop_seniors']

# Compute age proportions (raw proportions)
df['prop_youth'] = df['2024_pop_youth'] / df['total_population_2024']
df['prop_working'] = df['2024_pop_workingage'] / df['total_population_2024']
df['prop_seniors'] = df['2024_pop_seniors'] / df['total_population_2024']

# Option: Create binary buckets for non-linear effects using the median split
median_working = df['prop_working'].median()
median_seniors = df['prop_seniors'].median()
df['high_working'] = (df['prop_working'] > median_working).astype(int)
df['high_seniors'] = (df['prop_seniors'] > median_seniors).astype(int)

# -------------------------------
# 2. Use an Interpretable Income Measure
# -------------------------------
# Assume df['2024_income_total'] is the total income summed across groups.
# Compute income per capita, and express it in thousands.
df['income_per_capita_2024'] = df['2024_income_total'] / df['total_population_2024']
df['income_per_capita_2024_k'] = df['income_per_capita_2024'] / 1000

# -------------------------------
# 3. Adjust Commute Variables
# -------------------------------
# Instead of scaling commute time, compute commute time per capita (or as a percentage)
df['commute_time_per_capita'] = (df['2024_commute_time'] / df['total_population_2024']) * 100

# Note: We are removing the "commute_drivealone" variable entirely.

# -------------------------------
# 4. Define the Regression Predictors
# -------------------------------
# Here we exclude the old age index and commute drive-alone variable.
# We include our bucketed age measures, income per capita (in thousands), education index,
# adjusted commute time, and other relevant variables.
predictors = [
    'total_ev_chargers',         # Count of EV chargers
    'newreg_2024_cumsum',        # Cumulative new registrations for 2024
    'total_population_2024',     # Total population (raw)
    'high_working',              # Binary bucket for working-age proportion
    'high_seniors',              # Binary bucket for senior proportion
    'income_per_capita_2024_k',  # Income per capita (in thousands)
    '2024_edu_index',            # Education index
    'commute_time_per_capita',   # Commute time per capita (percentage)
    'commute_wfh',               # Work-from-home percentage (interpretable as is)
    'labor_unemployed'           # Labor unemployment percentage (or raw, as desired)
]

# -------------------------------
# 5. Prepare Data and Fit the Model
# -------------------------------
# Ensure there are no missing values in the selected predictors and dependent variable.
df_reg = df.dropna(subset=predictors + ['newreg_2024_elec_cumsum'])

# Add a constant for the intercept
X = sm.add_constant(df_reg[predictors])
y = df_reg['newreg_2024_elec_cumsum']

# Fit the OLS regression model
model = sm.OLS(y, X).fit()
print(model.summary())


NameError: name 'df' is not defined